# Начинаем запускать пайп

In [ ]:
from qdrant_client import QdrantClient
from typing import List, Dict, Any
import requests

Подключаемся к Qdrant/Infinity и готовим инструмент `retrieve`

In [ ]:
API_KEY = "MASHA_BIBA_BOBA"

INFINITY_API_URL = "http://localhost:7997"
EMBEDDINGS_ENDPOINT = f"{INFINITY_API_URL}/embeddings"
RERANK_ENDPOINT = f"{INFINITY_API_URL}/rerank"
EMBEDDING_MODEL = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
client = QdrantClient(
    url="http://localhost:6333",
    api_key=API_KEY,
    timeout=3000.0
)

COLLECTION_NAME = "rag_chunks_v3"
# COLLECTION_NAME = "rag_chunks_v2"


/tmp/ipykernel_786758/3638106874.py:15: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


In [24]:
def rerank_docs(docs: List[Any], query: str = "", api_key: str = API_KEY) -> List[Any]:
    if not docs or not query:
        return docs
    
    is_dict_format = isinstance(docs[0], dict) if docs else False
    
    if is_dict_format:
        # documents = [doc.get('content', '') if isinstance(doc, dict) else str(doc) for doc in docs]
        documents = [doc.get('document', '') if isinstance(doc, dict) else str(doc) for doc in docs]
    else:
        documents = [str(doc) for doc in docs]
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    
    response = requests.post(
        RERANK_ENDPOINT,
        json={
            "model": RERANK_MODEL, 
            "query": query,
            "documents": documents,
            "return_documents": True,
            "top_n": len(documents)
        },
        headers=headers,
        timeout=3000.0
    )
    
    response.raise_for_status()
    rerank_results = response.json()["results"]

    final_docs = []
    for result in rerank_results:
        original_index = result["index"]
        original_doc = docs[original_index]
        
        if is_dict_format:
            reranked_doc = {
                "document": result["document"],
                "source": original_doc.get("doc_url", "unknown"),
                "score": result["relevance_score"]
            }
        else:
            reranked_doc = {
                "document": result["document"], 
                "source": "unknown",
                "score": result["relevance_score"]
            }
        
        final_docs.append(reranked_doc)
    return final_docs

In [25]:
import pandas as pd

In [26]:
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
def get_query_embedding(query: str, api_key: str = API_KEY) -> List[float]:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    
    response = requests.post(
        EMBEDDINGS_ENDPOINT,
        json={
            "model": EMBEDDING_MODEL,
            "input": [query]
        },
        headers=headers,
        timeout=3000.0
    )
    
    response.raise_for_status()
    result = response.json()
    
    if isinstance(result, list) and len(result) > 0:
        if isinstance(result[0], list):
            return result[0]
        elif isinstance(result[0], dict) and 'embedding' in result[0]:
            return result[0]['embedding']
    elif isinstance(result, dict):
        if 'data' in result and len(result['data']) > 0:
            return result['data'][0].get('embedding', [])
        elif 'embedding' in result:
            return result['embedding']
        elif 'embeddings' in result and len(result['embeddings']) > 0:
            return result['embeddings'][0]
    
    if isinstance(result, list) and len(result) > 0 and isinstance(result[0], (int, float)):
        return result
    return result

## Обработка чанков

### PRODUCT_SYNONYMS / SYNONYMS_MAP:

задаём словари синонимов и разных написаний продуктов/терминов
Любая вариация (опечатки, латиница, дефисы) приводится к одной
канонической форме, чтобы запросы и документы искались стабильнее

`clean_chunk_text():`

нормализует текст чанка и заменяет все варианты написания продуктов
и ключевых слов на их канонический вид. 

Нужен, чтобы:
- убрать шум от опечаток и разных написаний,
- улучшить качество BM25 и векторного поиска за счёт единых токенов.


In [ ]:
import re
import unicodedata


PRODUCT_SYNONYMS = {
    'дебетовая альфа карта': [
        'дебетовая альфа-карта', 'дебет альфа карта', 'альфа дебет',
        'alfa debit card', 'alfa debit', 'alfa debet', 'debet alfa card',
        'дебетовая альфа карта', 'альфадебетовая карта', 'alfa debitcard'
    ],
    'кредитная альфа карта': [
        'кредитная альфа-карта', 'кредитная карта альфа', 'кредитка альфа',
        'alfa credit card', 'alfa credit', 'alfa creditcard',
        'кредитная карта альфа банк', 'credit alfa card', 'альфа кредитка'
    ],
    'альфа карта': [
        'альфа-карта', 'альфакарта', 'alfa card', 'alfa-card',
        'alfacard', 'alfa-carta', 'a карта', 'alfa карта'
    ],
    'альфа карта с гравировкой': [
        'альфа-карта с гравировкой', 'alfa карта с гравировкой',
        'alfacard engraving', 'карта с гравировкой', 'alfa гравировка карты'
    ],
    'платежный альфа стикер': [
        'платёжные альфа стикеры', 'альфа стикер', 'alfa sticker',
        'alfasticker', 'стикер alfa', 'платежный стикер альфа', 'платежный стикер'
    ],
    'карта alfa only': [
        'alfa only карта', 'карта альфа only', 'alfa only card',
        'alfa-only card', 'альфа only card'
    ],
    'детская карта': [
        'детская альфа карта', 'alfa kids card', 'child card', 'kids card',
        'карта для детей', 'карта для ребёнка', 'дет карта'
    ],
    'alfa travel': [
        'alfa travel card', 'альфа travel', 'alfa-тревел', 'альфа тревел',
        'alfa-travel карта', 'alfa travel карта', 'alfa trevel'
    ],
    'апельсиновая карта': [
        'оранжевая карта', 'apelsinovaya karta', 'alfa orange card',
        'оранжевая альфа карта', 'apelsin card', 'апельсинка'
    ],
    # Специальные условия
    'альфа карта для зарплаты': [
        'карта для зарплаты', 'альфа-карта для зарплаты', 'зарплатная альфа карта',
        'salary alfa card', 'salary card alfa'
    ],
    'альфа карта с комбо счётом': [
        'альфа карта с комбо-счётом', 'альфа карта combo',
        'alfa combocard', 'комбо карта альфа'
    ],
    'зарплата каждый день': [
        'зарплата каждый день', 'pay everyday', 'daily salary',
        'ежедневная зарплата', 'зарплата ежедневно'
    ],
    'подписка альфа смарт': [
        'альфа смарт', 'alfa smart', 'smart subscription', 'смарт подписка',
        'подписка smart', 'альфа smart подписка'
    ],
    # Вклады и счета
    'альфа вклад': [
        'альфа-вклад', 'alfa vklad', 'alfa deposit', 'альфа депозит',
        'alfa deposit', 'альфа вклады'
    ],
    'альфа вклад пенсионный': [
        'альфа-вклад пенсионный', 'пенсионный вклад альфа', 'alfa pension deposit',
        'alfa pension vklad', 'пенсионный депозит альфа'
    ],
    'альфа вклад для новых денег': [
        'альфа вклад для новых денег', 'новые деньги альфа вклад',
        'new money deposit alfa', 'alfa new money deposit', 'новые деньги вклад альфа'
    ],
    'социальный вклад': [
        'социальный вклад', 'social deposit', 'soc deposit', 'соц вклад'
    ],
    'альфа счет': [
        'альфа-счет', 'alfa schet', 'alfa account', 'альфа аккаунт',
        'alpha schet', 'alfa-account'
    ],
    'социальный счет': [
        'социальный счёт', 'social account', 'soc schet', 'соц счет'
    ],
    'копилка для зарплаты': [
        'зарплатная копилка', 'salary piggy bank', 'salary savings jar',
        'piggy bank', 'копилка зарплаты'
    ],
    'автопополнение накопительного счета': [
        'автопополнение счёта', 'auto top-up account', 'автопополнение',
        'auto replenish account', 'автопополнение накопительного счета'
    ],
    'драгоценные металлы': [
        'драг металлы', 'precious metals', 'драгоц металлы', 'metal investments'
    ],
    'детская копилка': [
        'детская копилка', 'alfa kids piggy', 'kids piggy bank', 'kids savings jar'
    ],
    'альфа вклад с программой долгосрочных сбережений': [
        'долгосрочные сбережения', 'long-term savings program',
        'long term savings programme', 'программа долгосрочных сбережений'
    ],

    # Кредиты
    'кредит наличными': [
        'наличный кредит', 'кредит наличный', 'cash loan', 'personal loan',
        'кредит на наличные', 'наличка кредит'
    ],
    'деньги до зарплаты': [
        'до зарплаты', 'payday loan', 'payday advance', 'займ до зарплаты',
        'деньги до получки', 'до зарплаты кредит'
    ],
    'кредит на автомобиль': [
        'кредит на авто', 'автокредит', 'auto loan', 'autocredit',
        'авто кредит', 'автокредитование'
    ],
    'кредит под залог': [
        'кредит под залог на любые цели', 'кредит под залог',
        'secured loan', 'loan secured', 'secured credit', 'заложенный кредит'
    ],
    'рефинансирование кредита': [
        'рефинансирование', 'перекредитование', 'refinancing', 'refinance',
        'рефинанс', 'перевод кредита', 'переоформление кредита'
    ],
    # Ипотека
    'ипотека': [
        'ипотечный кредит', 'mortgage loan', 'mortgage', 'ипотека альфа'
    ],
    'вторичное жильё': [
        'вторичка', 'вторичное жилье', 'resale mortgage',
        'resale property mortgage', 'вторичное жилье ипотека'
    ],
    'ипотека на дом': [
        'домовая ипотека', 'ипотека дом', 'house mortgage',
        'mortgage for house', 'ипотека для дома'
    ],
    'семейная ипотека': [
        'семейная ипотека', 'family mortgage', 'family home loan',
        'ипотека семейная', 'семейная жилищная ипотека'
    ],
    'ипотека на новостройку': [
        'новостройки', 'новостройка', 'новостройка ипотека',
        'new building mortgage', 'newbuild mortgage'
    ],
    'ипотека для ит специалистов': [
        'ипотека для it специалистов', 'ипотека для айти',
        'it mortgage', 'айти ипотека', 'ипотека для айти специалистов'
    ],
    'дальневосточная ипотека': [
        'дальневосточная ипотека', 'арктическая ипотека',
        'far eastern mortgage', 'arctic mortgage',
        'далневосточная и арктическая ипотека'
    ],
    'машино место ипотека': [
        'машино-место', 'машиноместо', 'parking space mortgage',
        'машиноместо ипотека', 'машино место'
    ],
    'коммерческая недвижимость': [
        'коммерческая недвижимость', 'commercial real estate mortgage',
        'коммерческая ипотека', 'business property mortgage'
    ],
    # Инвестиции
    'альфа инвестиции': [
        'альфа‑инвестиции', 'alfa investments', 'alfa invest', 'alfainvest',
        'alfa investicii', 'alfa investic', 'альфа инвестиция', 'альфа инвестирование'
    ],
    'счет для инвестиций': [
        'счёт для инвестиций', 'инвестиционный счёт', 'инвестиционный счет',
        'investment account', 'investment schet', 'инвест счёт'
    ],
    'каталог ценных бумаг': [
        'каталог бумаг', 'каталог акций', 'каталог облигаций',
        'catalog of securities', 'catalog securities', 'ценные бумаги каталог'
    ],
    'иис': [
        'индивидуальный инвестиционный счет', 'индивидуальный инвестиционный счёт',
        'индивидуальный инвест счёт', 'индивидуальный инвест счет', 'инвестиционный счет',
        'investment individual account', 'iic', 'iis'
    ],
    'обмен валюты на бирже': [
        'обмен валюты', 'валютный обмен', 'currency exchange', 'currency trading',
        'обмен валют', 'exchange currencies'
    ],
    'альфа форекс': [
        'alfa forex', 'alfa-forex', 'альфа форекс', 'alfa fx', 'alfa forex trading'
    ],
    'альфа турнир': [
        'альфа турнир', 'alfa tournament', 'alfa-турнир', 'турнир alfa',
        'alfa tournament 200 млн'
    ],
    'альфа трейдинг': [
        'alfa trading', 'alfa трейдинг', 'альфа трейдинг', 'alfa trade', 'alfa trading platform'
    ],
    'инвестиционное страхование жизни': [
        'исж', 'investment life insurance', 'life insurance investment',
        'invest life insurance', 'альфа инвестиционное страхование жизни'
    ],
    'стратегии': [
        'стратегии', 'strategies', 'investment strategies', 'стратегии иис',
        'strategy iis'
    ],
    'доверительное управление': [
        'trust management', 'доверительное управление', 'trust management investments',
        'доверительное управление активами'
    ],

    'мобильное приложение для инвестиций': [
        'приложение для инвестиций', 'alfa invest app', 'alfa инвестиции app',
        'invest app alfa', 'invest mobile app', 'мобильное приложение альфа инвестиции',
        'alfa investments mobile', 'мобайл alfa invest'
    ],
    'альфа инвестиции онлайн': [
        'alfa инвестиции онлайн', 'alfa invest online', 'alfa investment online',
        'alfa investments online', 'alfa investionline', 'альфа‑инвестиции онлайн'
    ],
    'веб терминал альфа инвестиции': [
        'веб-терминал альфа-инвестиции', 'web terminal alfa invest',
        'alfa investments web terminal', 'alfa invest web', 'веб терминал'
    ],
    'про терминал альфа инвестиции': [
        'pro терминал альфа инвестиции', 'pro-terminal alfa invest',
        'alfa investments pro terminal', 'pro terminal', 'альфа инвестиции pro терминал'
    ],
    # Премиальный сервис Alfa Only
    'alfa only': [
        'alfa-only', 'alfaonly', 'альфа only', 'альфа-only', 'alpha only',
        'alfa only сервис', 'альфа only сервис'
    ],
    'alfa only travel': [
        'alfa only travel', 'alfa only travel card', 'alfa travel only',
        'alfa travel only card', 'alfa only travel карта'
    ],
    'alfa only aeroflot': [
        'alfa only aeroflot', 'alfa only aeroflot card',
        'alfa-only aeroflot', 'альфа only aeroflot', 'аэрофлот карта alfa only'
    ],
    'карта alfa only mir supreme': [
        'alfa only mir supreme', 'alfa only mir supreme card',
        'mir supreme card', 'alfa mir supreme', 'альфа only mir supreme'
    ],
    'тонкий стикер alfa only': [
        'тонкий стикер', 'thin alfa sticker', 'thin sticker alfa only',
        'тонкий sticker', 'тонкий стикер alfa only'
    ],
    'стикер alfa only': [
        'alfa only sticker', 'sticker alfa only', 'alfa sticker only',
        'стикер альфа only'
    ],
    'стикер alfa only travel': [
        'стикер alfa only travel', 'alfa only travel sticker',
        'alfa sticker only travel', 'travel sticker alfa only'
    ],
    'платежное кольцо': [
        'платёжное кольцо', 'payment ring', 'alfa ring', 'кольцо альфа',
        'алфа кольцо', 'alfa pay ring'
    ],
    'alfa only lounge': [
        'alfa only lounge', 'alfa-only lounge', 'alfa lounge only',
        'alfa lounge', 'alfa only лаунж', 'alfa лаунж'
    ],
    'металлическая карта': [
        'metal card', 'metall card', 'metallic card', 'metal card alfa',
        'металлическая альфа карта', 'метал карта'
    ],
    'партнерская программа цум': [
        'партнёрская программа цум', 'alfa tsum partnership',
        'partner program tsum', 'tsum partnership', 'partner program alfa only'
    ],
    'премиальный вклад': [
        'premium deposit', 'premium vklad', 'alfa premium deposit',
        'альфа премиальный вклад', 'премиальный депозит'
    ],
    'smart тариф': [
        'смарт тариф', 'alfa smart', 'smart subscription', 'смарт план',
        'smart plan', 'alfa smart тариф'
    ],
    'like тариф': [
        'лайк тариф', 'like tarif', 'like plan', 'alfa like', 'like pack'
    ],
    'roaming тариф': [
        'роуминг', 'roaming', 'roaming tarif', 'tarif roaming', 'alfa роуминг', 'alfa roaming'
    ],
    # Самозанятость и семья
    'приложение для самозанятых': [
        'alfa self-employed app', 'self employed app', 'самозанятые приложение',
        'alfa se app', 'приложение альфа самозанятые'
    ],
    'ии помощник': [
        'ии-помощник', 'ai помощник', 'ai-помощник', 'ИИ помощник',
        'ai assistant', 'assistant AI', 'алфа помощник'
    ],
    'реклама для самозанятых': [
        'self-employed advertising', 'advertising for self-employed',
        'ads for self-employed', 'advertise for self-employed',
        'реклама самозанятых', 'реклама для самозанятого'
    ],
    'crm система': [
        'crm-система', 'crm system', 'crm', 'customer relationship management',
        'crm-система', 'alfa crm'
    ],
    'кредитные продукты для самозанятых': [
        'кредитные продукты', 'loan products for self-employed',

        'кредит самозанятым', 'кредиты для самозанятых',
        'loan for self-employed', 'кредитные продукты самозанятые'
    ],
    'налоги для самозанятых': [
        'налоги самозанятых', 'tax for self-employed', 'self-employed taxes',
        'taxes self-employed', 'налог для самозанятого', 'налоги самозанятого'
    ],
    'приложение для детей': [
        'kids app', 'child app', 'children app', 'alfa kids app',
        'детское приложение', 'приложение альфа дети'
    ],
    'детский стикер': [
        'kids sticker', 'детский альфа стикер', 'alfa kids sticker',
        'детский наклейка', 'sticker for kids'
    ],
    'семейный счет': [
        'семейный счёт', 'family account', 'family schet',
        'family deposit', 'семейный аккаунт', 'alfa family account'
    ],
    'карта для молодежи': [
        'карта для молодёжи', 'youth card', 'карта молодежная',
        'alfa youth card', 'альфа youth карта'
    ],
    'карта для пенсионеров': [
        'pensioner card', 'pension card', 'пенсионная карта',
        'alfa pension card', 'alfa пенсионная карта'
    ],
    # Образование, программы и сервисы
    'альфа будущее': [
        'alfafuture', 'alfa future', 'альфа будущее', 'альфабудущее',
        'alfa future program', 'alfa future platform'
    ],
    'а клуб': [
        'a-клуб', 'a club', 'alfa club', 'alphaclub', 'а клуб альфа', 'а-клуб'
    ],
    'магистратура': [
        'magistratura', 'masters programme', 'masters program',
        'алфа магистратура', 'альфа магистратура'
    ]
}

SYNONYMS_MAP = {
    'кешбэк': [
        'кэшбэк', 'кэшбек', 'кешбек', 'кеш бэк', 'кеш бек',
        'кэш бэк', 'кэш бек', 'кеш‑бэк', 'кеш‑бек',
        'кэш‑бэк', 'кэш‑бек', 'кебшек', 'кашбек', 'кашбэк',
        'cashback', 'cash back', 'cash-back', 'кажбэк'
    ],
    'альфа банк': [
        'альфабанк', 'альфа-банк', 'alfabank', 'alfa bank',
        'alfa-bank', 'alpha bank'
    ],
    'онлайн касса': [
        'онлайн-касса', 'онлайнкасса', 'online касса', 'online-касса',
        'онлайн кассы', 'онлайн-кассы', 'online кассы',
        'online-кассы', 'онлайнкаса'
    ],
    'ип': [
        'индивидуальный предприниматель', 'индивидуального предпринимателя',
        'индивидуальные предприниматели', 'индивидуальных предпринимателей',
        'индивидуальной предприниматель'
    ],
    'смс': [
        'sms', 'смс-уведомление', 'sms-уведомление',
        'смс-сообщение', 'sms-сообщение', 'смс сообщение', 'sms сообщение'
    ],
    'ккт': [
        'контрольно-кассовая техника', 'контрольно кассовая техника',
        'контрольно-кассовую технику', 'контрольно кассовую технику',
        'контрольно-кассовые', 'контрольно кассовые'
    ],
}

In [ ]:
import csv
import re
import unicodedata
from functools import lru_cache

In [ ]:

SYNONYM_MAP = {}
for canonical, forms in {**SYNONYMS_MAP, **PRODUCT_SYNONYMS}.items():
    all_forms = list(set(forms + [canonical]))
    SYNONYM_MAP[canonical] = all_forms

PATTERNS = []
for canonical, forms in SYNONYM_MAP.items():
    for form in sorted(forms, key=len, reverse=True):
        pattern = re.escape(form).replace(r"\ ", r"[ _-]*").replace(r"\-", r"[ _-]*")
        regex = re.compile(
            fr"(?:^|\W)({pattern})(?:\W|$)",
            flags=re.IGNORECASE | re.UNICODE
        )
        PATTERNS.append((regex, canonical))

@lru_cache(maxsize=1000)  
def clean_chunk_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    
    text = unicodedata.normalize('NFKC', text).lower().replace('ё', 'е')
    text = re.sub(r'[^а-яa-z0-9\s\-_]', ' ', text) 
    text = re.sub(r'[\s\-_]+', ' ', text).strip()   
    
    for regex, canonical in PATTERNS:
        def replace_match(match):
            start, end = match.start(1), match.end(1)
            return f"{text[match.start():start]}{canonical}{text[end:match.end()]}"
        
        text = regex.sub(replace_match, text)
    
    return re.sub(r'\s+', ' ', text).strip()


In [ ]:
def process_csv(input_path: str, output_path: str) -> None:
    with open(input_path, encoding='utf-8', newline='') as inp, \
         open(output_path, 'w', encoding='utf-8', newline='') as out:
        reader = csv.reader(inp)
        writer = csv.writer(out)
        header = next(reader)
        writer.writerow(header)
        for row in reader:
            if len(row) >= 2:
                row[1] = clean_chunk_text(row[1])
            writer.writerow(row)

process_csv('questions.csv', 'questions_ready.csv')

Можно потыкать

In [30]:
clean_chunk_text('кашбек альфа дебетовая карта')

'кешбэк альфа дебетовая карта'

## Обработка входных запросов

у нас очень много в запросах мусора и ошибок

Функция `preprocess` очищает текст запроса: нормализует его, убирает эмодзи, лишние пробелы и совсем бессмысленные строки.  
Все цифры заменяются на `0`, а по контексту числа превращаются в маски вроде `<телефон>`, `<дата>`, `<сумма>`, `<номер>`, `<код>` и т.д.  

Это нужно, чтобы модель (и при поиске) понимала, с чем имеет дело, а н промто 0000 


In [ ]:
import re
import unicodedata

_WHITESPACE_RE = re.compile(r"\s+")
_DIGIT_RE = re.compile(r"\d")

_MONTHS = (
    "января|февраля|марта|апреля|мая|июня|июля|августа|"
    "сентября|октября|ноября|декабря"
)

_EMOJI_PATTERN = re.compile(
    "["  # emoji ranges
    u"\U0001F600-\U0001F64F"
    u"\U0001F300-\U0001F5FF"
    u"\U0001F680-\U0001F6FF"
    u"\U0001F1E0-\U0001F1FF"
    u"\U0001F900-\U0001F9FF"
    u"\U0001FA70-\U0001FAFF"
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE,
)


def preprocess(query: str) -> str:
    if not query:
        return ""

    text = unicodedata.normalize("NFKC", query).strip()
    if not text:
        return ""

    text = _EMOJI_PATTERN.sub("", text)

    text = _WHITESPACE_RE.sub(" ", text)
    text = text.lower().replace("ё", "е")

    # минус бред (л.ЛЛл)
    letters_only = re.sub(r"[^a-zа-я]", "", text)
    if not letters_only:
        return "<непонятный запрос>"
    if len(letters_only) <= 4 and len(set(letters_only)) == 1:
        return "<непонятный запрос>"

    text = _DIGIT_RE.sub("0", text)

    # ===== ДАТЫ, ВРЕМЯ, ПЕРИОДЫ =====

    text = re.sub(r"\+0{10,}", "<телефон>", text)
    text = re.sub(r"(телефон[а-я]*\s+)[0\s\-()]{7,}", r"\1<телефон>", text)

    text = re.sub(r"\b0{2}[./]0{2}[./]0{4}\b", "<дата>", text)
    text = re.sub(r"\b0+\s+(" + _MONTHS + r")\b", r"<дата> \1", text)
    text = re.sub(r"\b(" + _MONTHS + r")\s+0{4}\b", r"\1 <год>", text)

    text = re.sub(r"\b0{4}\s*[-–]\s*0{4}\s*год[а-я]*\b", "<период>", text)
    text = re.sub(r"\b0{4}\s*[-–]\s*0{4}\b", "<период>", text)

    text = re.sub(
        r"\b0+\s*[-–]\s*0+\s*"
        r"(?:дн[еяй]|месяц[а-я]*|мес\.?|мес(?:яц[а-я]*)?|"
        r"недел[яеи]?|г[оа]д[а-я]*)\b",
        "<период>",
        text,
    )

    text = re.sub(
        r"\b0+\s*(?:дн[еяй]|месяц[а-я]*|мес\.?|месяцев|недел[яеи]?|лет|год[а-я]*)\b",
        "<период>",
        text,
    )

    text = re.sub(r"\b0{2}/0{4}\b", "<дата>", text)
    text = re.sub(r"\b0{1,2}:0{2}(?::0{2})?\b", "<время>", text)
    text = re.sub(r"\b0{1,2}[-–]0{2}\b", "<время>", text)
    text = re.sub(r"\b0+\s+числа\b", "<дата>", text)
    text = re.sub(r"\bминут[аы]?\s+0+\s+назад\b", "<период>", text)
    text = re.sub(r"\bдо\s+0{4}\s+гол[а-я]*\b", "<дата>", text)

    # ===== СУММЫ, ПРОЦЕНТЫ, ЛИМИТЫ =====

    text = re.sub(r"\b0+[.,]0{2,}\b", "<сумма>", text)
    text = re.sub(r"\b0{1,3}(?:\s+0{3})+\b", "<сумма>", text)

    text = re.sub(r"(сумм[ау]\s+на\s+)\b0+\b", r"\1<сумма>", text)
    text = re.sub(r"(сумма\s+кредит[а-я]*\s+)\b0+\b", r"\1<сумма>", text)
    text = re.sub(r"(кредит[а-я]*\s+на\s+)\b0+\b", r"\1<сумма>", text)
    text = re.sub(r"\b0+\b(?=\s+за\s+авто\s*пополнени[ея])", "<сумма>", text)

    currency_pattern = (
        r"(?:руб[а-я]*|rur|₽|usd|eur|тенге|сом|грн|uah|k|т\.р|тыс|тр\b|р\b|р\.|"
        r"доллар|евро)"
    )

    text = re.sub(
        r"\b0+\b(?=\s*" + currency_pattern + r")",
        "<сумма>",
        text,
    )

    text = re.sub(r"-0+\b", "-<сумма>", text)
    text = re.sub(r"(лимит[а-я]*\s+)\b0+\b", r"\1<лимит>", text)

    text = re.sub(
        r"(перевод[а-я]*\s+на\s+)\b0+\b(?!\s*" + currency_pattern + r")",
        r"\1<номер>",
        text,
    )

    text = re.sub(r"(оплат[а-я]*\s+)\b0+\b", r"\1<сумма>", text)
    text = re.sub(r"(перевел[а-я]*\s+)\b0+\b", r"\1<сумма>", text)

    text = re.sub(
        r"(смог[уае][а-я]*\s+только\s+)\b0+\b(?![.,])",
        r"\1<сумма>",
        text,
    )

    text = re.sub(
        r"(полож[а-я]*\s+)\b0+\b(?=\s+на\s+(?:карт[ауеы]|счет|счёт))",
        r"\1<сумма>",
        text,
    )

    text = re.sub(
        r"(отобража[ае][тся]*\s+)\b0+\b(?!\s*балл)",
        r"\1<сумма>",
        text,
    )
    text = re.sub(
        r"(показыва[аеют][тся]*\s+)\b0+\b(?!\s*балл)",
        r"\1<сумма>",
        text,
    )

    text = re.sub(r"\b0+\b(?=\s*балл[а-я]*\b)", "<число>", text)

    for prefix in [
        "сумм[ау]", "стоимост[ья]", "оплат[аи]", "платеж[ау]",
        "перевод", "пополнени[ея]", "кешбек", "кэшбэк", "кредит",
        "рассрочк[аи]", "долг", "остаток", "баланс",
        "коммунал", "комисси[ея]", "чек", "взнос",
        "поступлени[ея]", "операци[яй]",
    ]:
        text = re.sub(r"(" + prefix + r"\s+)\b0+\b", r"\1<сумма>", text)

    text = re.sub(r"0+\s*%", "<процент>", text)
    text = re.sub(r"\b0+\b\s+процент[а-я]*\b", "<процент>", text)

    text = re.sub(r"(фз\s+)\b0+\b", r"\1<номер>", text)

    # ===== НОМЕРА, КОДЫ, АДРЕСА =====

    text = re.sub(r"0\*0+|\*0+", "<номер>", text)
    text = re.sub(r"№\s*[a-zа-яё-]*0+", "<номер>", text)

    text = re.sub(
        r"\b(дом|д\.|квартира|кв\.|кв|корпус|корп\.|стр\.|стр|подъезд|под\.|к\.)\s*0+\.?",
        lambda m: m.group(1).strip() + " <номер>",
        text,
    )

    text = re.sub(
        r"\b(ул\.|улица|улице|улицы|улицу|улицей)\s+([^\d\s]+(?:\s+[^\d\s]+)*)\s+0+\b",
        r"\1 \2",
        text,
    )

    for cp in [
        "инн", "кпп", "огрн", "огрнип", "окпо", "снилс", "snils",
        "iban", "bic", "бик", "к/с", "корсчет", "корсчёт",
        "р/с", "рс", "рсч", "расчетн", "расчётн",
        "лицевой", "л/с", "октмо", "октмфл",
    ]:
        text = re.sub(r"(" + cp + r"\s+)[0\s-]+\b", r"\1<код>", text)

    for pref in [
        "номер", "карта", "карты", "карточки",
        "счет", "счёт", "счета", "счете",
        "договор", "договора", "расчетный", "расчётный",
    ]:
        text = re.sub(r"(" + pref + r"\s+)[0\s-]+\b", r"\1<номер>", text)

    text = re.sub(r"\b0+/0+\b", "<число>/<число>", text)

    # ===== ЧИСЛОВЫЕ КЕЙСЫ =====

    text = re.sub(r"(нет[ау]\s+)\b0+\b(?!\s*балл)", r"\1<сумма>", text)
    text = re.sub(r"\b0+\b(?=\s+нет[ау]\b)", "<сумма>", text)

    text = re.sub(r"\b0+\s+недел[яеи]\b", "<период>", text)

    text = re.sub(
        r"\b0+\b(?=\s*(?:перевод[а-я]*|операци[яй]|поход[а-я]*|визит[а-я]*|раз[а-я]*|граф[а-я]*))",
        "<число>",
        text,
    )

    text = re.sub(r"\b0+\b(?=\s+карт[ауеы])", "<число>", text)
    text = re.sub(r"\b0+\b(?=\.)", "<число>", text)
    text = re.sub(r"\b[xх]{3,}(?:/[a-zа-я]+)?\b", "<данные>", text)

    # Финальный фолбэк для чисел
    text = re.sub(r"\b0+\b", "<число>", text)

    # Финальная нормализация пробелов
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


In [32]:
def process_csv(input_path: str, output_path: str) -> None:
    with open(input_path, encoding='utf-8', newline='') as inp, \
         open(output_path, 'w', encoding='utf-8', newline='') as out:
        reader = csv.reader(inp)
        writer = csv.writer(out)
        header = next(reader)
        writer.writerow(header)
        for row in reader:
            if len(row) >= 2:
                row[1] = preprocess(row[1])
            writer.writerow(row)

process_csv('questions_ready.csv', 'questions_fin.csv')

# Сам Retriev

гибридный поиск

In [33]:
from qdrant_client import models
from fastembed import SparseTextEmbedding


sparse_encoder = SparseTextEmbedding(model_name="Qdrant/bm25")

def _encode_sparse(text: str) -> models.SparseVector | None:
    cleaned = (text or "").strip()
    if not cleaned:
        return None
    embedding = next(sparse_encoder.embed([cleaned]))
    return models.SparseVector(indices=embedding.indices.tolist(), values=embedding.values.tolist())



def hybrid_search(
    query: str,
    collection_name: str = COLLECTION_NAME,
    vector_weight: float = 0.5,
    text_weight: float = 0.3,
    limit: int = 20,
    score_threshold: float = 0.0
) -> List[Dict[str, Any]]:
    ppcd_query = preprocess(query)
    
    if not ppcd_query:
        return []

    try:
        # query_dense_vector = embedder.get_query_embedding(ppcd_query)
        query_dense_vector = get_query_embedding(ppcd_query, api_key=API_KEY)
    except Exception as e:
        print(f"Ошибка получения эмбеддинга: {e}")
        return []

    try:
        query_sparse_vector = _encode_sparse(ppcd_query)
    except Exception as e:
        print(f"⚠️ Ошибка получения эмбеддинга: {e}")
        return []
    
    all_results = {}
    
    try:
        vector_results = client.query_points(
            collection_name=collection_name,
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            prefetch=[
                models.Prefetch(
                    query=query_dense_vector,
                    using="dense",  
                    limit=limit * 4, 
                ),

                models.Prefetch(
                    query=query_sparse_vector,
                    using="bm25",
                    limit=limit * 4, 
                ),
            ],
            limit=limit * 2,  
            score_threshold=score_threshold,
            with_payload=True,
            with_vectors=False
        )
        vector_results = vector_results.points
        max_vector_score = max([r.score for r in vector_results], default=1.0)
        for result in vector_results:
            doc_id = result.id
            normalized_score = (result.score / max_vector_score) * vector_weight if max_vector_score > 0 else 0
            
            if doc_id not in all_results:
                all_results[doc_id] = {
                    'id': doc_id,
                    'score': normalized_score,
                    'vector_score': result.score,
                    'text_score': 0.0,
                    'payload': result.payload
                }
            else:
                all_results[doc_id]['score'] += normalized_score
                all_results[doc_id]['vector_score'] = result.score
    except Exception as e:
        print(f"упали на векторном поиске: {e}")
    
    sorted_results = sorted(
        all_results.values(),
        key=lambda x: x['score'],
        reverse=True
    )

    formatted_docs = []
    for result in sorted_results[:limit]:
        payload = result.get('payload', {})
        formatted_docs.append({
            'id': result['id'],
            'score': result['score'],
            'doc_url': payload.get('doc_url', 'unknown'),
            'content': payload.get('chunk_text', ''),
            'document': payload.get('chunk_text', ''),
            'metadata': {k: v for k, v in payload.items() if k not in ['doc_url', 'chunk_text']}
        })

    
    return formatted_docs

In [ ]:
# @tool
def retrieve(query: str, top_k: int = 10) -> str:
    """
    Tool(уже не тул, думали агента делать раньше) для получения необходимой информации из базы знаний.
    """
    if not query or not query.strip():
        return "Результат работы инструмента: Пустой запрос"
    
    search_results = hybrid_search(
        query=query,
        limit=top_k * 2,  
        vector_weight=0.7,
        text_weight=0.3
    )
    
    if not search_results:
        return "Результат работы: Документы не найдены"
    
    final_docs = rerank_docs(search_results, query=query)
    final_docs = final_docs[:top_k]


    result = "Результат:\n\n"
    for i, doc in enumerate(final_docs, 1):
        source = doc.get('source', 'unknown')
        content = doc.get('document', '')
        
        result += f"source: ```{source}```\n\n"
        result += f"content: {content}\n\n"
        result += f"---\n\n"
    
    return result

import asyncio
from typing import List, Dict, Any, Optional

async def retrieve_async(query: str, top_k: int = 10) -> str:
    """
    Асинхронная версия
    """
    if not query or not query.strip():
        return "Результат работы инструмента: Пустой запрос"
    
    try:
        loop = asyncio.get_running_loop()
        
        search_results = await loop.run_in_executor(
            None, 
            lambda: hybrid_search(
                query=query,
                limit=top_k * 2,
                vector_weight=0.6,
            )
        )
        
        if not search_results:
            return "Результат работы: Документы не найдены"

        final_docs = await loop.run_in_executor(
            None,
            lambda: rerank_docs(search_results, query=query)
        )
        
        final_docs = final_docs[:top_k]
        result = "Результат:\n\n"
        for i, doc in enumerate(final_docs, 1):
            source = doc.get('source', 'unknown')
            content = doc.get('document', '')
            
            result += f"source: ```{source}```\n\n"
            result += f"content: {content}\n\n"
            result += f"---\n\n"
        
        return result
    
    except asyncio.CancelledError:
        raise
    except Exception as e:
        error_msg = f"Ошибка при поиске: {str(e)}"
        print(f"{error_msg}")  
        return f"Результат работы: {error_msg}. Повторите запрос позже."


In [37]:
test_query = "бик"
result = retrieve(test_query, top_k=15)
print(result)

Результат:

source: ```https://alfabank.servicecdn.ru/site-upload/f4/c4/1869/dogovor_cbo_1072025.pdf?previewDocument=true```

content: номер счета бик при наличии у клиента открытого счета ов мир в валюте российской федерации предоставить оператору есиа его их реквизиты наименование счета номер счета бик об исполнении поручения банк информирует клиента посредством направления зм сообщения на номер телефона сотовой связи клиента зарегистрированный в системах банка поручение исполняется при наличии соответствующей технической возможности при исполнении обязанностей предусмотренных настоящим пунктом банк руководствуется требованиями законодательства российской федерации 3 5 операции по переводу денежных средств со счета осуществляются исключительно на основании заявления поручения и или распоряжения клиента в том числе длительного распоряжения поручения оформленного по установленной банком форме

---

source: ```https://alfabank.servicecdn.ru/site-upload/e3/30/1869/dogovor_cbo_1082025.pdf

agent

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# agent_instructions = open('prompts/agentic_prompt.md', 'r', encoding='utf-8').read()

llm = ChatOpenAI(
    api_key='API_KEY',
    base_url='http://localhost:11434/v1',
    model="ai-sage/GigaChat3-10B-A1.8B",
    temperature=0,
    max_tokens = 1000,
)


# agent_instructions = open('prompts/agentic_prompt.md', 'r', encoding='utf-8').read()
llm.invoke([HumanMessage('privet')])


AIMessage(content='\n\nПривет! 😊 Как я могу помочь?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 10, 'total_tokens': 101, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 't-tech/T-pro-it-2.0', 'system_fingerprint': None, 'id': 'chatcmpl-0caf74a1-94b4-4d77-aa8c-b9d16f994a32', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019af4e1-c6c0-79b3-968d-d0e7401d54db-0', usage_metadata={'input_tokens': 10, 'output_tokens': 91, 'total_tokens': 101, 'input_token_details': {}, 'output_token_details': {}})

In [ ]:
import asyncio
from typing import List, Dict, Any, Optional
from langchain_core.messages import SystemMessage, HumanMessage
import csv
import threading
import langchain
import nest_asyncio
import random  
nest_asyncio.apply()


langchain.verbose = True

csv_lock = threading.Lock()
csv_initialized = False

In [ ]:
async def async_call_with_retry(
    coro_factory,
    max_retries: int = 3,
    base_delay: float = 0.5,
    max_delay: float = 8.0,
    retry_exceptions: tuple[type[BaseException], ...] = (Exception,),
    context_name: str = ""
):
    """
    экспоненциальный ретрай
    """
    attempt = 0
    while True:
        try:
            return await coro_factory()
        except retry_exceptions as e:
            attempt += 1
            if attempt > max_retries:
                print(f"[{context_name}] Исчерпаны попытки ({max_retries}), ошибка: {e}")
                raise
            delay = min(base_delay * (2 ** (attempt - 1)), max_delay)
            jitter = delay * 0.1
            sleep_for = delay + random.uniform(0, jitter)
            print(
                f"[{context_name}] Ошибка: {e}. "
                f"Попытка {attempt}/{max_retries}, спим {sleep_for:.2f} сек."
            )
            await asyncio.sleep(sleep_for)


def init_csv_file(filename: str = 'submission_realtime.csv'):
    global csv_initialized
    with csv_lock:
        if not csv_initialized:
            with open(filename, 'w', encoding='utf-8', newline='') as csvfile:
                writer = csv.writer(csvfile, delimiter='\t')
                writer.writerow(['q_id', 'query', 'answer', 'status'])
            csv_initialized = True



In [ ]:
async def write_result_to_csv(
    q_id: int,
    query: str,
    answer: str,
    status: str = 'success',
    filename: str = 'submission_realtime.csv'
):
    """блудем писать сразу, чтоб не потерять ничего """
    try:
        with csv_lock:
            with open(filename, 'a', encoding='utf-8', newline='') as csvfile:
                writer = csv.writer(csvfile, delimiter='\t')
                writer.writerow([q_id, query, answer, status])
        return True
    except Exception as e:
        print(f"Ошибка записи в CSV для q_id={q_id}: {str(e)}")
        return False


async def retrieve_with_retry(search_query: str, max_retries: int = 3) -> Any:
    """
    Обёртка над retrieve
    """
    async def _call():
        if hasattr(retrieve_async, '__await__'):
            return await retrieve_async(search_query)
        else:
            return retrieve(search_query)

    return await async_call_with_retry(
        _call,
        max_retries=max_retries,
        base_delay=0.5,
        max_delay=8.0,
        context_name="retrieve"
    )

### тут самое важное и промпты

In [ ]:
async def function_unified_collect_alpha_async(query: str, max_iter=3) -> str:

    if llm is None:
        raise ValueError("LLM модель не определена. Пожалуйста, определите переменную 'llm'")

    search_res = []
    query = preprocess(query)
    try:
        for iteration in range(max_iter):
            pair = [
                SystemMessage(content="""Ты — эксперт по поиску информации в базе знаний Альфа-Банка.
Твоя задача:
На основе предобработанного запроса клиента сделать ОДИН качественный поисковый запрос для внутреннего поиска по базе знаний.

На вход приходит:
- Текст запроса клиента.
- В тексте могут быть маски в угловых скобках: <сумма>, <дата>, <номер>, <лимит> и другие.

Правила:
1. Тематика:
   - Всегда думай, что речь идёт только об Альфа-Банке.
   - Поисковый запрос должен относиться к продуктам и услугам Альфа-Банка.

2. Маски:
   - Всегда сохраняй маски <...> как есть.

3. Качество запроса:
   - Исправляй ошибки, если они мешают поиску.
   - Используй банковские термины и ключевые слова: кредит, карта, счёт, перевод, комиссия, рассрочка, кредитная карта и т.п.
   - Если исходный запрос слишком общий, уменьшь его до одной наиболее важной и понятной темы.
   - Можно использовать синонимы и более точные формулировки, чтобы запрос лучше подходил для поиска.

4. Формат ответа:
   - Верни РОВНО ОДНУ поисковую фразу.
   - Без кавычек, без пояснений, без дополнительного текста.
   - Короткая фраза, а не длинное предложение или абзац."""),

                HumanMessage(content=f"""Исходный запрос пользователя: "{query}"

РАНЕЕ ВЫПОЛНЕННЫЕ ПОИСКОВЫЕ ЗАПРОСЫ И ИХ РЕЗУЛЬТАТЫ:
{"\n".join(search_res) if search_res else "Пока нет выполненных запросов"}

ТВОЯ ЗАДАЧА: Придумай {['ПЕРВЫЙ', 'ВТОРОЙ', 'ТРЕТИЙ'][iteration]} НОВЫЙ поисковый запрос для базы знаний""")
            ]

            search_response = await async_call_with_retry(
                lambda: llm.ainvoke(pair),
                max_retries=3,
                base_delay=0.5,
                max_delay=8.0,
                context_name="llm_search_query"
            )
            search_query = search_response.content.strip()

            search_query = search_query.strip('"').strip("'").strip()

            try:
                search_results = await retrieve_with_retry(search_query, max_retries=3)
            except Exception as e:
                search_results = f"КРИТИЧЕСКАЯ ОШИБКА ПОИСКА: {str(e)}"

            search_res.append(
                f'ЗАПРОС #{iteration + 1}: "{search_query}"\n'
                f'РЕЗУЛЬТАТЫ ПОИСКА:\n{search_results}\n{"=" * 50}'
            )

        pair = [
            SystemMessage(content="""Ты — сотрудник поддержки Альфа-Банка.  
Твоя задача — отвечать на вопросы клиентов ТОЛЬКО по информации из предоставленного контекста.

Входные данные:
- Вопрос клиента (может содержать ошибки или быть неформальным).  
- Контекст — фрагменты из официальной базы знаний банка.  
- В тексте могут встречаться маски: `<сумма>`, `<дата>`, `<номер>` и т.п. - — это скрытые реальные данные клиента

Правила ответа:
1. Сначала ПОНЯТНО И ЧЕТКО определи, о чём спрашивает клиент (даже если вопрос написан с ошибками).  
2. Если в контексте есть хотя бы частично релевантная информация — используй её:  
   - Можно перефразировать, объединить фрагменты, обобщить.  
   - Не выдумывай и не добавляй того, чего нет в контексте.  
3. Если в контексте нет ничего, что относится к вопросу — верни ровно одну фразу: `Эталонного ответа нет` 
4. Если даёшь ответ, он должен быть кратким, ясным и полезным — от 2 до 7 предложений
"""),
            HumanMessage(content=f"""ЗАПРОС ПОЛЬЗОВАТЕЛЯ: "{query}"

КОНТЕКСТ ИЗ БАЗЫ ЗНАНИЙ (РЕЗУЛЬТАТЫ ПОИСКА):

{"\n\n".join(search_res)}""")
        ]

        final_response = await async_call_with_retry(
            lambda: llm.ainvoke(pair),
            max_retries=3,
            base_delay=0.5,
            max_delay=8.0,
            context_name="llm_final_answer"
        )
        final_answer = final_response.content.strip()

        return final_answer

    except Exception as e:
        error_msg = f"КРИТИЧЕСКАЯ ОШИБКА ОБРАБОТКИ ЗАПРОСА '{query}': {str(e)}"
        print(error_msg)
        return "Ответа нет в базе знаний Альфа-Банка."

In [ ]:
async def run_agent_async(
    query: str,
    q_id: int,
    semaphore: Optional[asyncio.Semaphore] = None
) -> Dict[str, Any]:
    try:
        if semaphore:
            async with semaphore:
                try:
                    result = await function_unified_collect_alpha_async(query)
                except Exception:
                    result = await function_unified_collect_alpha_async(query, max_iter=2)
        else:
            result = await function_unified_collect_alpha_async(query)

        await write_result_to_csv(q_id, query, result, 'success')

        return {
            'q_id': q_id,
            'query': query,
            'result': result,
            'status': 'success'
        }
    except Exception as e:
        error_message = str(e)
        await write_result_to_csv(q_id, query, f"Ошибка: {error_message}", 'failed')
        return {
            'q_id': q_id,
            'query': query,
            'error': error_message,
            'status': 'failed'
        }


async def run_multiple_queries_parallel(
    queries: List[str],
    max_concurrent: int = 10,
    timeout: float = 30.0,
    filename: str = 'submission_realtime.csv'
) -> List[Dict[str, Any]]:
    """
    параллелка
    """
    init_csv_file(filename)

    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = []

    for idx, query in enumerate(queries, 1): 
        task = asyncio.create_task(
            asyncio.wait_for(
                run_agent_async(query, idx, semaphore),
                timeout=timeout
            )
        )
        tasks.append(task)

    results = await asyncio.gather(*tasks, return_exceptions=True)

    processed_results = []
    for i, result in enumerate(results):
        q_id = i + 1
        query = queries[i]

        if isinstance(result, Exception):
            error_msg = f"Timeout or execution error: {str(result)}"
            await write_result_to_csv(q_id, query, f"Ошибка: {error_msg}", 'failed')
            processed_results.append({
                'q_id': q_id,
                'query': query,
                'error': error_msg,
                'status': 'failed'
            })
        else:
            processed_results.append(result)

    return processed_results

In [ ]:
async def run_in_notebook(
    queries: List[str],
    max_concurrent: int = 15,
    timeout: float = 45.0,
    show_sample_results: int = 5,
    filename: str = 'submission_realtime.csv'
):
    init_csv_file(filename)

    results = await run_multiple_queries_parallel(
        queries=queries,
        max_concurrent=max_concurrent,
        timeout=timeout,
        filename=filename
    )

    success_count = sum(1 for r in results if r['status'] == 'success')
    failed_count = len(results) - success_count
    print(f"\nСтатистика выполнения:")
    print(f"Успешных: {success_count}")
    print(f"Неудачных: {failed_count}")

    print(f"\nВсе результаты успешно записаны в '{filename}'")
    return results


def run_async_in_notebook(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        return loop.run_until_complete(coro)
    else:
        if loop.is_running():
            import nest_asyncio
            nest_asyncio.apply()
            task = asyncio.create_task(coro)
            return loop.run_until_complete(task)
        else:
            return loop.run_until_complete(coro)

In [ ]:
import pandas as pd
queries = pd.read_csv('questions_ready.csv')['query'].tolist()[0:]
print("=== ЗАПУСК В РЕЖИМЕ JUPYTER NOTEBOOK ===")
results = await run_in_notebook(
    queries=queries,
    max_concurrent=35, 
    timeout=30000.0,
    show_sample_results=3,
    filename='submission_realtime.csv'
)

print(f"\nОбработка завершена. Всего обработано запросов: {len(results)}")

=== ЗАПУСК В РЕЖИМЕ JUPYTER NOTEBOOK ===


Сборка сабмишена

In [ ]:
import pandas as pd

file_path = 'submission_realtime.csv'
df = pd.read_csv(file_path, delimiter='\t', header=None)

df.columns = ['q_id', 'question', 'answer', 'status']

df_clean = df.drop_duplicates(subset='q_id', keep='first').copy()

df_clean['q_id'] = pd.to_numeric(df_clean['q_id'], errors='coerce').astype('Int64')

df_clean = df_clean.dropna(subset=['q_id'])


min_id = df_clean['q_id'].min()
max_id = df_clean['q_id'].max()

full_ids = pd.DataFrame({'q_id': range(min_id, max_id + 1)})

final_df = full_ids.merge(df_clean[['q_id', 'answer']], on='q_id', how='left')
final_df['answer'] = final_df['answer'].fillna("Эталонного ответа нет")

final_df = final_df.sort_values('q_id').reset_index(drop=True)

final_df[['q_id', 'answer']].to_csv('submission_vector_v2_86.csv', index=False)

# УСЕ